# Logistic Loss & Cost – Practice Skeleton

**Short name (GitHub):** `Logistic_Loss_Cost`  
**Lab source:** Coursera Machine Learning Specialization · C1_W3 Lab04 (Logistic Loss) + Lab05 (Cost Function)  
**Language:** Python (NumPy + Matplotlib)

Use this notebook to practice. Open **`Logistic_Loss_Cost_Solution.ipynb`** only after you attempt each exercise.  
Companion files: `Logistic_Loss_Cost_Cheatsheet.docx`, `Logistic_Loss_Cost_Reusable_Template.ipynb`, `logistic_loss_cost_flowchart.png`.

### Learning objectives
- Explain why squared error + sigmoid is a poor training objective
- Implement per-example **logistic loss** (piecewise and compact forms)
- Implement the **binary cross-entropy cost** $J(w,b)$ with a loop and a vectorized alternate
- Compare two candidate parameter sets by cost (and by the decision boundary)
- Run a small Monte-Carlo simulation (sample size, label noise, overconfident predictions)
- Adapt the same numbers for an analyst, an executive, and a non-specialist reader

### Data files
- `data/logistic_loss_1d.csv` — tumor-size style 1-D labels
- `data/logistic_loss_2d.csv` — 6-point 2-D lab example
- `data/logistic_loss_practice.csv` — extra 2-D practice set


## Inline cheat-sheet (keep this cell visible)

See also **`Logistic_Loss_Cost_Cheatsheet.docx`**.

| Item | Formula / code |
|------|----------------|
| Sigmoid | $g(z)=1/(1+e^{-z})$ · `1/(1+np.exp(-np.clip(z,-50,50)))` |
| Model | $f=g(w\cdot x+b)$ |
| Loss $y=1$ | $-\log(f)$ |
| Loss $y=0$ | $-\log(1-f)$ |
| Compact loss | $-\big[y\log f+(1-y)\log(1-f)\big]$ |
| Cost | $J=(1/m)\sum_i L_i$ |
| Stability | clip $f$ to `[1e-15, 1-1e-15]` before `log` |
| Squared error (do **not** use) | $\frac{1}{2m}\sum(f-y)^2$ — non-convex with a sigmoid |

**Flow of the lab:** data → $z$ → sigmoid $f$ → per-example loss → mean cost $J$ → compare $(w,b)$.


## 0. Packages


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline
np.set_printoptions(precision=6, suppress=True)
print("Libraries loaded")



## 1. Why squared error fails for logistic regression

Linear regression uses
$$
J(w,b)=\frac{1}{2m}\sum_{i=0}^{m-1}\big(f_{w,b}(x^{(i)})-y^{(i)}\big)^2,
\qquad f_{w,b}(x)=wx+b.
$$
That surface is a smooth “soup bowl”, so gradient descent is reliable.

For logistic regression $f_{w,b}(x)=g(wx+b)$ with $g$ the sigmoid. The composition of a square with a sigmoid is **no longer a bowl**. Local wiggles and flat regions appear, which makes plain gradient descent unreliable.

### Task 1.1 — load the 1-D example and plot labels
`size_cm` is a single feature; `malignant` is 0/1.


In [ ]:
# TODO: load data/logistic_loss_1d.csv into x_1d (shape (m,)) and y_1d (shape (m,))
# Hint: arr = np.loadtxt("data/logistic_loss_1d.csv", delimiter=",", skiprows=1)

# YOUR CODE HERE


print("m =", x_1d.shape[0])
print("labels:", y_1d)



### Task 1.2 — scatter the 1-D labels
Plot size on the x-axis and the 0/1 label on the y-axis. Use different markers for the two classes.


In [ ]:
# TODO: scatter plot of x_1d vs y_1d
# YOUR CODE HERE



### Task 1.3 — squared-error cost on a sigmoid model

Implement
$$
J_{\text{sq}}(w,b)=\frac{1}{m}\sum_i \big(g(wx^{(i)}+b)-y^{(i)}\big)^2.
$$
Then evaluate it on a small grid of $(w,b)$ and print the min / max you observe. You should see that the surface is not a simple bowl (inspect `logistic_vs_squared_heatmap.png` after you compute a few values).


In [ ]:
def sigmoid(z):
    """Stable sigmoid. Works on scalars and ndarrays."""
    ### START CODE HERE ###
    
    ### END CODE HERE ###


def squared_cost_logistic(x, y, w, b):
    """Mean squared error of sigmoid(w*x+b) vs y."""
    ### START CODE HERE ###
    
    ### END CODE HERE ###


# quick check
print("sigmoid(0) should be 0.5 ->", sigmoid(0.0))
print("J_sq at (w,b)=(2.2,-4.4) =", squared_cost_logistic(x_1d, y_1d, 2.2, -4.4))
print("J_sq at (w,b)=(0.6,-0.4)  =", squared_cost_logistic(x_1d, y_1d, 0.6, -0.4))



## 2. Logistic loss (one example)

**Loss** = error on a *single* training example.  
**Cost** = average loss over the training set.

Piecewise definition:
$$
L\big(f_{w,b}(x^{(i)}), y^{(i)}\big)=
\begin{cases}
-\log\big(f_{w,b}(x^{(i)})\big) & y^{(i)}=1\\
-\log\big(1-f_{w,b}(x^{(i)})\big) & y^{(i)}=0
\end{cases}
$$

Compact form (easier to code):
$$
L = -\Big[ y\log(f) + (1-y)\log(1-f) \Big].
$$

When the prediction $f$ matches the label, $L\to 0$. When $f$ is confident *and wrong*, $L\to\infty$.

### Task 2.1 — implement both forms
Clip $f$ to `[1e-15, 1-1e-15]` so `log(0)` never fires.


In [ ]:
def logistic_loss_piecewise(f, y):
    """f, y may be scalars. Return the scalar loss."""
    ### START CODE HERE ###
    
    ### END CODE HERE ###


def logistic_loss_compact(f, y):
    """f, y may be arrays. Return an array of losses (or a scalar)."""
    ### START CODE HERE ###
    
    ### END CODE HERE ###


# unit checks
for f, y in [(0.9, 1), (0.1, 0), (0.9, 0), (0.1, 1)]:
    a = logistic_loss_piecewise(f, y)
    b = float(logistic_loss_compact(np.array([f]), np.array([y])))
    print(f"f={f}, y={y}: piecewise={a:.4f}  compact={b:.4f}")



### Task 2.2 — draw the two loss curves
For $f\in(0,1)$ plot $-\log(f)$ and $-\log(1-f)$. Confirm:
- $y=1$ curve is 0 at $f=1$ and blows up as $f\to 0$
- $y=0$ curve is 0 at $f=0$ and blows up as $f\to 1$


In [ ]:
# TODO: plot the two logistic-loss curves
# YOUR CODE HERE



## 3. Cost function $J(w,b)$

$$
J(w,b)=\frac{1}{m}\sum_{i=0}^{m-1} L\big(f_{w,b}(x^{(i)}), y^{(i)}\big)
$$
with $f=g(z)$, $z=w\cdot x+b$.

### Task 3.1 — loop implementation (matches the original lab)
`X` has shape $(m,n)$ or $(m,)$ for the 1-D case. Handle both.


In [ ]:
def compute_cost_logistic(X, y, w, b):
    """
    Loop form of binary cross-entropy.
    X: (m,) or (m,n)
    y: (m,)
    w: scalar or (n,)
    b: scalar
    """
    ### START CODE HERE ###
    
    ### END CODE HERE ###


# 1-D check: a well-separated fit should beat a weak fit
print("1-D J(2.2, -4.4) =", compute_cost_logistic(x_1d, y_1d, 2.2, -4.4))
print("1-D J(0.6, -0.4)  =", compute_cost_logistic(x_1d, y_1d, 0.6, -0.4))



### Task 3.2 — vectorized alternate
No Python `for` over examples. Use `X @ w` (2-D) or `w * X` (1-D).


In [ ]:
def compute_cost_logistic_vec(X, y, w, b):
    """Vectorized binary cross-entropy. Same signature as the loop version."""
    ### START CODE HERE ###
    
    ### END CODE HERE ###


print("vec 1-D J(2.2, -4.4) =", compute_cost_logistic_vec(x_1d, y_1d, 2.2, -4.4))
print("loop vs vec close?",
      np.isclose(compute_cost_logistic(x_1d, y_1d, 2.2, -4.4),
                 compute_cost_logistic_vec(x_1d, y_1d, 2.2, -4.4)))



## 4. Two-feature lab example (C1_W3 Lab05)

Load `data/logistic_loss_2d.csv`. Compare
- $w=(1,1),\; b=-3$  → boundary $x_0+x_1=3$
- $w=(1,1),\; b=-4$  → boundary $x_0+x_1=4$

The original lab reports
$J(-3)\approx 0.3669$ and $J(-4)\approx 0.5037$.
The worse visual fit must have the **higher** cost.


In [ ]:
# TODO: load X_2d (m,2) and y_2d (m,)
# YOUR CODE HERE


w_tmp = np.array([1.0, 1.0])
print("J b=-3:", compute_cost_logistic(X_2d, y_2d, w_tmp, -3))
print("J b=-4:", compute_cost_logistic(X_2d, y_2d, w_tmp, -4))



### Task 4.1 — plot both decision boundaries on the 2-D points


In [ ]:
# TODO: scatter the two classes; overlay x1 = 3-x0 and x1 = 4-x0
# YOUR CODE HERE



## 5. More practice

### 5.1 Three predictions, three labels
Compute the compact loss for each pair and the mean cost.

| # | f | y |
|---|---|---|
| 1 | 0.95 | 1 |
| 2 | 0.20 | 0 |
| 3 | 0.80 | 0 |


In [ ]:
# TODO: losses and mean cost for the table above
# YOUR CODE HERE



### 5.2 Extra 2-D file
Load `data/logistic_loss_practice.csv`. Evaluate $J$ at
$w=(1.0, 0.8),\, b=-0.4$ and at $w=(0,0),\, b=0$.
Which pair is better? Why is $w=0,b=0$ a useful baseline?


In [ ]:
# TODO
# YOUR CODE HERE



### 5.3 Optional numeric-stability drill
Compute `-np.log(0.0)` and then the clipped version. What would happen inside gradient descent without clipping?


In [ ]:
# TODO
# YOUR CODE HERE



## 6. Simulation — change a few knobs, watch $J$ move

The cost of a *fixed* $(w,b)$ changes when the data-generating process changes.
Edit the boxed parameters and re-run.

What you should see:
- larger $m$ → $J$ estimated at the true parameters settles (smaller Monte-Carlo spread)
- higher label-flip rate → $J$ rises (the model is “right” but the labels lie)
- pushing predictions toward 0 or 1 on the *wrong* side → $J$ explodes


In [ ]:
# ===== editable simulation knobs =====
TRUE_W = np.array([1.4, 1.1])
TRUE_B = -0.2
M_LIST = [20, 40, 80, 160, 320]
NOISE_LIST = [0.00, 0.05, 0.10, 0.20, 0.30]
N_REPS = 20
SEED = 7
# =====================================

def make_dataset(m, noise, seed):
    rng = np.random.default_rng(seed)
    X = rng.normal(size=(m, 2))
    p = sigmoid(X @ TRUE_W + TRUE_B)
    y = (rng.random(m) < p).astype(float)
    nflip = int(noise * m)
    if nflip:
        idx = rng.choice(m, size=nflip, replace=False)
        y[idx] = 1 - y[idx]
    return X, y

# TODO: for each m in M_LIST, run N_REPS datasets with noise=0.05
#       record mean and std of compute_cost_logistic_vec(X, y, TRUE_W, TRUE_B)
# TODO: for each noise in NOISE_LIST, fix m=120 and do the same
# TODO: plot two error-bar charts

# YOUR CODE HERE



### 6.1 Overconfidence sweep
Hold data fixed. For one example with $y=1$, let $f$ run from 0.01 to 0.99 and plot the loss. Repeat for $y=0$. This is the same pair of curves as Task 2.2 — now treat it as a *policy* warning: a model that is 99% sure and wrong is far more expensive than a model that is 60% sure and wrong.


In [ ]:
# TODO
# YOUR CODE HERE



## 7. Audience notes (write 3–5 sentences each)

Use the numbers you computed. Do **not** paste formulas into the executive or non-specialist versions.

1. **Analyst / technician** — mention $J$, the two candidate $(w,b)$, and why the loss is unbounded for confident mistakes.
2. **Executive** — one headline (“the tighter boundary is the better scoring rule”), the two cost numbers, and what you would do next (fit $w,b$ by descending $J$).
3. **Non-specialist** — a tumor-size or exam-score story with no jargon; “how surprised the rule is when it is wrong”.


In [ ]:
analyst_note = """
YOUR TEXT
"""

executive_note = """
YOUR TEXT
"""

nonspecialist_note = """
YOUR TEXT
"""

print(analyst_note)
print(executive_note)
print(nonspecialist_note)



## 8. Flowchart of the desired outcome

Open `logistic_loss_cost_flowchart.png` (also rendered below if the file is in the working directory).

The next lab in the sequence takes $\partial J/\partial w$ and $\partial J/\partial b$ and runs gradient descent. Do **not** skip the loss/cost lab — a convex, numerically stable $J$ is the reason GD works.


In [ ]:
from IPython.display import Image, display
import os
for p in ["logistic_loss_cost_flowchart.png",
          "logistic_loss_curves.png",
          "logistic_vs_squared_heatmap.png"]:
    if os.path.exists(p):
        display(Image(p, width=640))
    else:
        print("missing", p)



## Recap checklist
- [ ] Sigmoid implemented and stable
- [ ] Piecewise loss matches compact loss
- [ ] Loop cost matches vectorized cost
- [ ] $J(b=-3) < J(b=-4)$ on the 2-D lab set
- [ ] Simulation knobs change the plots
- [ ] Three audience paragraphs written
